# Poor Man's Lakehouse — Demo Notebook

This notebook demonstrates the key features of the project. It walks through:

1. **Catalog Browsing** — explore namespaces and tables without a JVM
2. **PyIceberg Client** — schema inspection, snapshot history, Polars scans
3. **Ibis Multi-Engine** — read/write Iceberg tables via DuckDB, PySpark, Polars
4. **Polars Client** — SQL queries with `%%sql` magic
5. **PySpark** — direct Spark access via catalog-specific builders

## Prerequisites

Start the Lakekeeper catalog before running this notebook:

```bash
just up lakekeeper
```

Make sure your `.env` has `CATALOG=lakekeeper`.

---
## 1. Catalog Browsing

The `CatalogBrowser` provides a quick, JVM-free way to explore what's in your catalog.
It auto-resolves the REST catalog URI from your `CATALOG` setting.

In [ ]:
from poor_man_lakehouse import CatalogBrowser

browser = CatalogBrowser()
print(f"Catalog: {browser}")
print(f"Namespaces: {browser.list_namespaces()}")

In [ ]:
# List tables in the default namespace
tables = browser.list_tables("default")
print(f"Tables in 'default': {tables}")

# If tables exist, inspect one
if tables:
    schema = browser.get_table_schema("default", tables[0])
    print(f"\nSchema of '{tables[0]}':")
    for col in schema:
        print(f"  {col['name']:20s} {col['type']:15s} required={col['required']}")

---
## 2. PyIceberg Client

The `PyIcebergClient` gives full access to Iceberg table metadata — schema evolution,
snapshot history, time travel — plus scanning to Polars or Arrow. No JVM required.

In [ ]:
from poor_man_lakehouse import PyIcebergClient

iceberg = PyIcebergClient()
print(iceberg)
print(f"Namespaces: {iceberg.list_namespaces()}")
print(f"Tables in 'default': {iceberg.list_tables('default')}")

In [ ]:
# If you have tables, inspect schema and history
tables = iceberg.list_tables("default")
if tables:
    table_name = tables[0][1]  # (namespace, name) tuple
    print(f"\n--- Schema of '{table_name}' ---")
    for col in iceberg.table_schema("default", table_name):
        print(f"  {col['name']:20s} {col['type']}")

    print(f"\n--- Snapshot History ---")
    for snap in iceberg.snapshot_history("default", table_name):
        print(f"  ID: {snap['snapshot_id']}  ts: {snap['timestamp_ms']}  ops: {snap['summary'].get('operation', 'N/A')}")

In [ ]:
# Scan a table to Polars LazyFrame (lazy evaluation)
if tables:
    table_name = tables[0][1]
    lf = iceberg.scan_to_polars("default", table_name)
    print(f"LazyFrame schema: {lf.collect_schema()}")
    display(lf.head(5).collect())

---
## 3. Ibis Multi-Engine Connection

The `IbisConnection` lets you access the same Iceberg tables through PySpark, Polars,
and DuckDB — all via a unified Ibis interface. DuckDB 1.5+ also supports **writing**
to Iceberg tables.

In [ ]:
from poor_man_lakehouse import IbisConnection

conn = IbisConnection()
print(f"Catalog: {conn._catalog_name}")
print(f"Endpoint: {conn._lakekeeper_endpoint}")

In [ ]:
# Create a table and write data via DuckDB
conn.create_table("default", "demo_users", "id INTEGER, name VARCHAR, age INTEGER")
print("Table created!")

# Insert rows
conn.write_table(
    "default", "demo_users", "duckdb",
    query="SELECT 1 AS id, 'Alice' AS name, 30 AS age UNION ALL SELECT 2, 'Bob', 25 UNION ALL SELECT 3, 'Charlie', 35"
)
print("Data written!")

In [ ]:
# Read with DuckDB
result = conn.sql("SELECT * FROM lakekeeper.default.demo_users", "duckdb")
print("--- DuckDB read ---")
print(result.execute())

In [ ]:
# Read the same table with Polars (via PyIceberg bridge)
polars_table = conn.read_table("default", "demo_users", "polars")
print("--- Polars read ---")
print(polars_table.execute())

In [ ]:
# Clean up
conn.close()

---
## 4. Polars Client with Lakekeeper Backend

The `PolarsClient` provides a SQL interface using Polars. With `backend="lakekeeper"`,
it uses PyIceberg under the hood to scan tables.

In [ ]:
from poor_man_lakehouse import PolarsClient

client = PolarsClient(backend="lakekeeper")
print(f"Catalogs: {client.list_catalogs()}")
print(f"Namespaces: {client.list_namespaces('lakekeeper')}")
print(f"Tables: {client.list_tables('lakekeeper', 'default')}")

In [ ]:
# Scan a table directly
lf = client.scan_table("lakekeeper", "default", "demo_users")
display(lf.collect())

In [ ]:
# Load the %%sql magic for Jupyter
from poor_man_lakehouse import load_sql_magic

load_sql_magic(client)

In [ ]:
%%sql
SELECT * FROM lakekeeper.default.demo_users
WHERE age > 25
ORDER BY name

---
## 5. PySpark Direct Access

For full Spark ecosystem access, use the catalog-specific builders.

In [ ]:
from poor_man_lakehouse import get_spark_builder, CatalogType

builder = get_spark_builder(CatalogType.LAKEKEEPER)
spark = builder.get_spark_session()
print(f"Spark version: {spark.version}")
print(f"Catalog: {spark.catalog.currentCatalog()}")

In [ ]:
# Query the table we created earlier
spark.sql("SELECT * FROM lakekeeper.default.demo_users").show()

In [ ]:
# Insert more data via Spark
spark.sql("""
    INSERT INTO lakekeeper.default.demo_users
    VALUES (4, 'Diana', 28), (5, 'Eve', 32)
""")
spark.sql("SELECT * FROM lakekeeper.default.demo_users ORDER BY id").show()

In [ ]:
spark.stop()

---
## Summary

| Connector | JVM Required | Read | Write | Best For |
|-----------|:---:|:---:|:---:|----------|
| `CatalogBrowser` | No | Metadata only | No | Quick exploration |
| `PyIcebergClient` | No | Polars, Arrow | No | Schema, snapshots, scans |
| `IbisConnection` | PySpark only | All 3 engines | DuckDB | Multi-engine comparison |
| `PolarsClient` | No | Polars | No | SQL + `%%sql` magic |
| `SparkBuilder` | Yes | PySpark | PySpark | Full Spark ecosystem |
| `DremioConnection` | No | Arrow Flight | No | Query federation |